In [ ]:
# Celda 1: imports, seeds y configuracion
# Balance: damaged y undamaged tienen la misma cantidad de imagenes en
# cada split. Se usan todas las undamaged de deteccion, y el cupo de
# damaged (la misma cantidad) se reparte entre dos fuentes:
#   - PROP_CLASIF de ese cupo sale del dataset de clasificacion
#   - el resto sale de las damaged de deteccion (no se usan todas, solo
#     las que hagan falta, elegidas al azar)
# Ej: con 3.000 undamaged y PROP_CLASIF = 0.3 -> 900 de clasificacion +
# 2.100 de deteccion = 3.000 damaged. Subir PROP_CLASIF si el detector
# sigue sin reconocer los danos del clasificador.
# BATCH_SIZE 16 y sin .cache() por el mismo problema de memoria RAM que
# en el notebook original.
# Etiquetas igual que antes: damaged = 0, undamaged = 1 (orden alfabetico
# de image_dataset_from_directory), asi la salida sigmoid sigue siendo la
# probabilidad hacia "undamaged" y el codigo de prediccion no cambia.

import os
import gc
import hashlib
import random
import subprocess

import numpy as np
import pandas as pd
import tensorflow as tf
import seaborn as sns
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

LOCAL_DATA_DIR = "data"
BUCKET = "s3://humanoid-damage-cnn-834257582282-us-east-1-an/processed/damage_detection"

DATASET_CSV = "data_clasificacion/dataset_final.csv"
PROP_CLASIF = 0.3

IMG_SIZE = (224, 224)
BATCH_SIZE = 16

CLASS_NAMES = ["damaged", "undamaged"]
MODEL_PATH = "../models/modelo_detection2.keras"

In [ ]:
# Celda 2: sincronizacion de imagenes de deteccion desde S3
# Igual que en el notebook original: "sync" solo copia, las imagenes del
# bucket nunca se tocan, y si ya estan descargadas solo baja lo que falta.
# Depende de: LOCAL_DATA_DIR, BUCKET (celda 1)

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

subprocess.run(["aws", "s3", "sync", f"{BUCKET}/damaged/", f"{LOCAL_DATA_DIR}/damaged/", "--quiet"], check=True)
subprocess.run(["aws", "s3", "sync", f"{BUCKET}/undamaged/", f"{LOCAL_DATA_DIR}/undamaged/", "--quiet"], check=True)

n_damaged = len(os.listdir(f"{LOCAL_DATA_DIR}/damaged"))
n_undamaged = len(os.listdir(f"{LOCAL_DATA_DIR}/undamaged"))
print(f"Danadas: {n_damaged} | No danadas: {n_undamaged}")

In [ ]:
# Celda 3: verificar el dataset de clasificacion
# Chequea que exista lo que genera cnn_damage_classification_build_dataset.ipynb
# (dataset_final.csv con image_path y label) y que las imagenes esten en
# disco. Si falta algo corta con un error que dice que correr antes.
# Depende de: DATASET_CSV (celda 1)

if not os.path.exists(DATASET_CSV):
    raise FileNotFoundError(f"No existe {DATASET_CSV}: correr antes cnn_damage_classification_build_dataset.ipynb")

clasif_df = pd.read_csv(DATASET_CSV)

faltan_columnas = {"image_path", "label"} - set(clasif_df.columns)
if faltan_columnas:
    raise ValueError(f"Al csv le faltan las columnas {sorted(faltan_columnas)}")

no_estan = [r for r in clasif_df["image_path"] if not os.path.exists(r)]
if no_estan:
    raise FileNotFoundError(f"{len(no_estan)} de {len(clasif_df)} imagenes no estan en disco (ej: {no_estan[0]})")

print(f"Dataset de clasificacion OK: {len(clasif_df)} imagenes, todas en disco\n")
print(clasif_df["label"].value_counts().to_string())

In [ ]:
# Celda 4: imagenes de deteccion y split train/val/test (70/15/15)
# En vez de image_dataset_from_directory se arma un df con las rutas,
# porque hay que mezclar dos fuentes de imagenes. El split es el mismo
# esquema que antes (70% train, el 30% restante a la mitad entre val y
# test), estratificado por clase con train_test_split. Se hace sobre
# TODAS las imagenes de deteccion; despues la celda 7 elige cuales damaged
# se usan dentro de cada split.
# Ojo: no es exactamente el mismo split que el del notebook original, asi
# que el test de este notebook no se puede usar para evaluar best_model.keras
# (tendria imagenes que ese modelo vio en train).
# Tambien calcula el hash md5 de cada archivo, que se usa en la celda 6
# para no agregar fotos de clasificacion que ya estan en deteccion.
# Solo se toman extensiones que TensorFlow puede leer.
# Depende de: LOCAL_DATA_DIR, CLASS_NAMES, SEED (celda 1)

FORMATOS_OK = (".jpg", ".jpeg", ".png", ".gif", ".bmp")

def md5_archivo(ruta):
    with open(ruta, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

filas = []
for clase in CLASS_NAMES:
    carpeta = f"{LOCAL_DATA_DIR}/{clase}"
    for archivo in sorted(os.listdir(carpeta)):
        if archivo.lower().endswith(FORMATOS_OK):
            filas.append({"image_path": f"{carpeta}/{archivo}", "clase": clase,
                          "origen": "deteccion", "grupo": f"{clase} (deteccion)"})
det_df = pd.DataFrame(filas)

det_train, det_resto = train_test_split(det_df, test_size=0.30, stratify=det_df["clase"], random_state=SEED)
det_val, det_test = train_test_split(det_resto, test_size=0.50, stratify=det_resto["clase"], random_state=SEED)
det_df["split"] = "train"
det_df.loc[det_val.index, "split"] = "val"
det_df.loc[det_test.index, "split"] = "test"

det_df["hash"] = det_df["image_path"].map(md5_archivo)

print("Imagenes de deteccion disponibles:")
print(pd.crosstab(det_df["clase"], det_df["split"])[["train", "val", "test"]])

In [ ]:
# Celda 5: cuantas imagenes damaged se toman de cada fuente
# En cada split: cupo de damaged = cantidad de undamaged. De ese cupo,
# PROP_CLASIF sale de clasificacion y el resto de deteccion.
# Si no alcanzan las damaged de deteccion para completar el cupo, corta
# con un error (en ese caso subir PROP_CLASIF).
# Depende de: det_df (celda 4), PROP_CLASIF (celda 1)

SPLITS = ["train", "val", "test"]

cupos = pd.DataFrame(index=SPLITS)
cupos["undamaged"] = det_df[det_df["clase"] == "undamaged"]["split"].value_counts().reindex(SPLITS)
cupos["damaged_clasif"] = (cupos["undamaged"] * PROP_CLASIF).round().astype(int)
cupos["damaged_deteccion"] = cupos["undamaged"] - cupos["damaged_clasif"]
cupos["deteccion_disponibles"] = det_df[det_df["clase"] == "damaged"]["split"].value_counts().reindex(SPLITS)

faltan = cupos[cupos["damaged_deteccion"] > cupos["deteccion_disponibles"]]
if not faltan.empty:
    raise ValueError(f"No alcanzan las damaged de deteccion en {list(faltan.index)}: subir PROP_CLASIF")

print(cupos)

In [ ]:
# Celda 6: imagenes de clasificacion que se agregan como damaged
# Rearma el split del entrenamiento de clasificacion (mismo csv y mismos
# train_test_split con random_state=42 que cnn_damage_classification_multiclass_training),
# asi train del clasificador -> train del detector, val -> val, test -> test.
# Las imagenes de test del clasificador nunca caen en train del detector.
# En cada split toma solo las que indica el cupo de la celda 5 (no todo el
# dataset de clasificacion):
#   1. arma una muestra un poco mas grande que el cupo (20% extra),
#      estratificada por superclase, y le calcula el hash md5
#   2. saca las que son la misma foto que alguna de deteccion (podrian
#      quedar en train por un lado y en test por el otro, o como damaged
#      y undamaged a la vez) y las repetidas entre si
#   3. de lo que queda toma exactamente el cupo, al azar
# Todas entran con clase "damaged"; la superclase se guarda en "grupo"
# para despues ver en que tipo de dano se equivoca el detector.
# Depende de: clasif_df (celda 3), det_df (celda 4), cupos (celda 5), SEED

c_train, c_resto = train_test_split(clasif_df, test_size=0.30, stratify=clasif_df["label"], random_state=42)
c_val, c_test = train_test_split(c_resto, test_size=0.50, stratify=c_resto["label"], random_state=42)

hashes_deteccion = set(det_df["hash"])
partes = []
for split, parte in zip(SPLITS, [c_train, c_val, c_test]):
    n = cupos.loc[split, "damaged_clasif"]
    frac = min(1.0, n * 1.2 / len(parte))
    candidatas = parte.groupby("label", group_keys=False).sample(frac=frac, random_state=SEED).copy()
    candidatas["hash"] = candidatas["image_path"].map(md5_archivo)
    repetidas = candidatas["hash"].isin(hashes_deteccion)
    candidatas = candidatas[~repetidas].drop_duplicates("hash")
    if len(candidatas) < n:
        raise ValueError(f"{split}: quedaron {len(candidatas)} imagenes de clasificacion y hacen falta {n}")
    muestra = candidatas.sample(n=n, random_state=SEED)
    print(f"{split}: {repetidas.sum()} candidatas descartadas por estar ya en deteccion, se toman {n}")
    partes.append(pd.DataFrame({
        "image_path": muestra["image_path"].values,
        "clase": "damaged",
        "origen": "clasificacion",
        "grupo": muestra["label"].values,
        "split": split,
    }))
extra_df = pd.concat(partes, ignore_index=True)

print(f"\nSe agregan {len(extra_df)} de {len(clasif_df)} imagenes de clasificacion como damaged\n")
print(pd.crosstab(extra_df["grupo"], extra_df["split"])[SPLITS])

In [ ]:
# Celda 7: dataset combinado y balanceado
# Toma todas las undamaged de deteccion y, en cada split, solo la cantidad
# de damaged de deteccion que indica el cupo (al azar). Las damaged de
# deteccion que sobran no se usan. Se une con las de clasificacion y se
# separa train/val/test. Label: damaged = 0, undamaged = 1.
# Al final chequea que damaged y undamaged tengan la misma cantidad en
# cada split; como esta balanceado no hacen falta class weights.
# Depende de: det_df (celda 4), cupos (celda 5), extra_df (celda 6)

det_usadas = [det_df[det_df["clase"] == "undamaged"]]
for split in SPLITS:
    disponibles = det_df[(det_df["clase"] == "damaged") & (det_df["split"] == split)]
    det_usadas.append(disponibles.sample(n=cupos.loc[split, "damaged_deteccion"], random_state=SEED))

todo_df = pd.concat(det_usadas + [extra_df], ignore_index=True).drop(columns="hash")
todo_df["label"] = todo_df["clase"].map({c: i for i, c in enumerate(CLASS_NAMES)})

train_df = todo_df[todo_df["split"] == "train"].reset_index(drop=True)
val_df = todo_df[todo_df["split"] == "val"].reset_index(drop=True)
test_df = todo_df[todo_df["split"] == "test"].reset_index(drop=True)

print(pd.crosstab([todo_df["clase"], todo_df["origen"]], todo_df["split"], margins=True)[SPLITS + ["All"]])

por_clase = pd.crosstab(todo_df["clase"], todo_df["split"])
assert (por_clase.loc["damaged"] == por_clase.loc["undamaged"]).all(), "damaged y undamaged no quedaron iguales"
print("\nOK: misma cantidad de damaged y undamaged en cada split")

In [ ]:
# Celda 8: pipeline de imagenes con tf.data
# Lee y redimensiona cada imagen a 224x224, en 0-255 igual que
# image_dataset_from_directory (el Rescaling 1/255 va dentro del modelo).
# Train se mezcla; val y test NO, asi el orden de las predicciones
# coincide con el de val_df/test_df y se pueden cruzar con origen y grupo.
# Depende de: train_df, val_df, test_df (celda 7), IMG_SIZE, BATCH_SIZE, SEED

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    return img, label

def make_dataset(dataframe, training):
    paths = dataframe["image_path"].values
    labels = dataframe["label"].values.astype("float32").reshape(-1, 1)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(dataframe), seed=SEED)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)
print(f"train: {len(train_df)} | val: {len(val_df)} | test: {len(test_df)}")

In [ ]:
# Celda 9: data augmentation
# Igual que en el notebook original: flip, rotacion, zoom y contraste.
# Va dentro del modelo, asi que solo actua durante el entrenamiento, no en
# validacion/test ni en predicciones nuevas.

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.15),
])

In [ ]:
# Celda 10: arquitectura (crear el modelo desde cero)
# IMPORTANTE: solo correr para empezar a entrenar de cero. Si ya existe
# modelo_detection2.keras y se quiere seguir entrenando, usar la celda 11.
# Es la misma CNN del notebook original (4 bloques conv 32->64->128->256
# con BatchNormalization y MaxPooling, GlobalAveragePooling, Dense,
# Dropout y salida sigmoid). Se entrena desde cero y no a partir de
# best_model.keras porque el split cambio: arrancar de ese modelo haria
# que el test tenga imagenes que ya vio.
# Depende de: data_augmentation (celda 9)

def build_cnn(input_shape=(224, 224, 3)):
    inputs = tf.keras.Input(shape=input_shape)

    x = layers.Rescaling(1./255)(inputs)
    x = data_augmentation(x)

    # Bloque 1
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    # Bloque 2
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    # Bloque 3
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    # Bloque 4
    x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)

    outputs = layers.Dense(1, activation="sigmoid")(x)

    return models.Model(inputs, outputs)

model = build_cnn()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")]
)
model.summary()

In [ ]:
# Celda 11: cargar el checkpoint (en vez de la celda 10)
# Para seguir entrenando despues de que se corto el kernel, o para solo
# evaluar. Trae el modelo ya compilado, con el estado del optimizador.
# Antes hay que correr las celdas 1 a 9 para tener los datasets.
# Depende de: MODEL_PATH (celda 1)

model = tf.keras.models.load_model(MODEL_PATH)
print(f"Modelo cargado desde {MODEL_PATH}")

In [ ]:
# Celda 12: callbacks
# Mismos que en el notebook original, con dos cambios:
#   - el checkpoint guarda el menor val_loss. En el original monitoreaba
#     val_recall, pero con damaged = 0 la metrica Recall de Keras mide el
#     recall de la clase 1 (undamaged), no el de damaged. El recall de
#     damaged se mira en la evaluacion (celda 14).
#   - se guarda en MODEL_PATH (modelo_detection2.keras), no en best_model.keras.
# MemoryCleanup fuerza garbage collection al final de cada epoca para
# evitar que el kernel se muera por memoria.
# Nota: EarlyStopping no tiene memoria entre tandas de fit(), su patience
# se reinicia en cada tanda; cuando parar se decide mirando si val_loss
# sigue mejorando entre tandas.

class MemoryCleanup(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        gc.collect()

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4),
    tf.keras.callbacks.ModelCheckpoint(MODEL_PATH, monitor="val_loss", mode="min", save_best_only=True),
    MemoryCleanup()
]

In [ ]:
# Celda 13: entrenamiento en tandas de 5 epocas
# Correr esta celda varias veces hasta que val_loss deje de mejorar entre
# tandas. Si el kernel se corta, correr celdas 1-9, la 11 (cargar
# checkpoint), la 12 y seguir desde aca.
# Depende de: model (celda 10 u 11), train_ds, val_ds, callbacks

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks
)

In [ ]:
# Celda 14: evaluacion en test
# Carga el mejor checkpoint y predice todo el test. Como test_ds no se
# mezcla, las predicciones quedan en el mismo orden que test_df.
# Muestra:
#   - reporte por clase, matriz de confusion y ROC-AUC (igual que antes)
#   - recall de damaged separado por origen y por superclase: cuantas de
#     las imagenes de cada grupo el modelo detecto como danadas. Es lo que
#     dice si el detector ya reconoce los danos del clasificador.
# Depende de: test_ds, test_df (celdas 7 y 8), MODEL_PATH, CLASS_NAMES

model = tf.keras.models.load_model(MODEL_PATH)

y_pred_prob = model.predict(test_ds, verbose=0).ravel()
y_true = test_df["label"].values
y_pred = (y_pred_prob > 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Matriz de confusion: CNN deteccion de danos v2")
plt.xlabel("Prediccion")
plt.ylabel("Real")
plt.tight_layout()
plt.show()

print(f"ROC-AUC: {roc_auc_score(y_true, y_pred_prob):.4f}\n")

res_df = test_df.assign(pred=np.array(CLASS_NAMES)[y_pred], prob_undamaged=y_pred_prob)
res_df["acierto"] = res_df["pred"] == res_df["clase"]
por_grupo = res_df.groupby(["origen", "grupo"]).agg(imagenes=("acierto", "size"), acierto=("acierto", "mean"))
por_grupo["acierto"] = (por_grupo["acierto"] * 100).round(1)
print("% de aciertos por grupo (para los grupos damaged es el recall de damaged):")
print(por_grupo.to_string())

In [ ]:
# Celda 15: ver predicciones sobre imagenes de test
# Misma idea que prediccion_clasificacion2: toma N_POR_GRUPO imagenes al
# azar de cada grupo del test (undamaged, damaged de deteccion y cada
# superclase de clasificacion) y muestra la foto con la clase real y la
# predicha (verde si acerto, rojo si no). Cada vez que se corre salen
# imagenes distintas. Al final imprime cuantas acerto.
# Depende de: res_df (celda 14)

N_POR_GRUPO = 2

a_probar = res_df.groupby("grupo", group_keys=False).apply(lambda g: g.sample(min(N_POR_GRUPO, len(g))))

aciertos = 0
for _, fila in a_probar.iterrows():
    img, _ = load_image(fila["image_path"], 0)
    prob_damaged = 1 - fila["prob_undamaged"]
    aciertos += fila["acierto"]

    plt.figure(figsize=(4, 4))
    plt.imshow(img.numpy().astype("uint8"))
    plt.axis("off")
    plt.title(f"{fila['grupo']}\nreal: {fila['clase']} | pred: {fila['pred']} (damaged {prob_damaged:.2%})",
              color="green" if fila["acierto"] else "red", fontsize=9)
    plt.show()

print(f"Acerto {aciertos} de {len(a_probar)} ({aciertos / len(a_probar):.0%})")

In [ ]:
# Celda 16: prediccion sobre imagenes nuevas
# Igual que en el notebook original pero con modelo_detection2. Recorre
# las imagenes de FOLDER_PATH y muestra la prediccion de cada una.
# La salida sigmoid es la probabilidad hacia "undamaged".
# Depende de: MODEL_PATH, IMG_SIZE (celda 1), load_image (celda 8)

FOLDER_PATH = "prueba"

model = tf.keras.models.load_model(MODEL_PATH)

archivos = sorted(f for f in os.listdir(FOLDER_PATH) if f.lower().endswith((".jpg", ".jpeg", ".png")))
print(f"Se encontraron {len(archivos)} imagenes en '{FOLDER_PATH}'\n")

for archivo in archivos:
    img, _ = load_image(os.path.join(FOLDER_PATH, archivo), 0)
    pred_prob = model.predict(tf.expand_dims(img, 0), verbose=0)[0][0]
    pred_label = "damaged" if pred_prob < 0.5 else "undamaged"

    plt.imshow(img.numpy().astype("uint8"))
    plt.axis("off")
    plt.title(f"{archivo} - Prediccion: {pred_label} (prob undamaged: {pred_prob:.2%})")
    plt.show()